# Hierarchical Risk Parity with Nonlinear Latent Covariance Filtering

**Author:** Gonzalo Ramírez-Carrillo

---

### Project Overview
This repository implements an advanced asset allocation framework that combines Marcos López de Prado's Hierarchical Risk Parity (HRP) protocol with deep learning architectures (Autoencoders). The primary objective is to enhance portfolio robustness out-of-sample by filtering estimation noise from historical covariance matrices through dynamic, nonlinear latent representations under a walk-forward optimization framework.

### ENVIRONMENT LIBRARIES & SYSTEM IMPORTS

In [3]:
# Core Data Science Libraries
import pandas as pd
import numpy as np
import random

# Hierarchical Clustering & Distance Metrics
from scipy.cluster.hierarchy import linkage
from sklearn.preprocessing import StandardScaler

# Deep Learning Architecture (TensorFlow / Keras)
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping

In [4]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import warnings
warnings.filterwarnings("ignore")

tf.get_logger().setLevel("ERROR")

### Global Reproducibility Configuration

In [6]:
def set_deterministic_seeds(seed=42):
    np.random.seed(seed)
    random.seed(seed)
    tf.random.set_seed(seed)
    # Configure Keras internal deterministic behavior
    tf.config.experimental.enable_op_determinism()
    print(f"[SYSTEM] Global seed successfully locked on determinism: {seed}")

set_deterministic_seeds(42)

[SYSTEM] Global seed successfully locked on determinism: 42


## 1. Data Ingestion & Preprocessing Pipeline

In [8]:
# Data
raw_prices = pd.read_excel('df_stocks.xlsx', sheet_name = 'df', index_col='Dates')
print('Number of shares:', raw_prices.shape[1])

Number of shares: 54


In [9]:
raw_prices.tail(3)

,BAP PE Equity,SCCO PE EQUITY,CHILE CI EQUITY,SQM/B CI EQUITY,PFCIBEST CB EQUITY,CENCOSUD CI EQUITY,BSAN CI EQUITY,LTM CI EQUITY,BCI CI EQUITY,CIBEST CB EQUITY,...,IAM CI EQUITY,ILC CI EQUITY,CAP CI EQUITY,CORFICOL CB EQUITY,SALFACOR CI EQUITY,FORUS CI EQUITY,SONDA CI EQUITY,OROB CI EQUITY,VOLCABC1 PE EQUITY,BICE CI EQUITY
Dates,,,,,,,,,,,,,,,,,,,,,
2026-04-09,353.0,188.50,171.99,73361.0,67880,2560.0,78.00,23.79,61500.0,85980,...,1000.00,20700.0,6815.0,16780.0,1334.9,2298.8,298.57,11.14,0.70,379.13
2026-04-10,349.0,189.01,173.00,73750.0,67460,2627.0,78.52,23.82,62605.0,86980,...,992.89,20850.0,6810.0,16700.0,1317.3,2289.0,303.89,11.38,0.70,393.34
2026-04-13,357.7,195.00,172.05,78100.0,69300,2565.0,78.58,23.59,62292.0,88200,...,995.00,20300.0,6945.0,16700.0,1298.2,2251.3,302.01,11.69,0.69,406.00


## Asset Returns and Data Filtering

In quantitative finance, **asset returns** represent the relative change in a financial instrument's price over a specified time interval. For this framework, we utilize **continuous log returns**, which offer superior mathematical properties for time-series modeling and neural network integration:

$$r_t = \log(P_t) - \log(P_{t-1}) = \log\left(\frac{P_t}{P_{t-1}}\right)$$

Where $P_t$ represents the asset price at time $t$. 

**Methodological Note on Asset Selection:**
Broad market indices (such as the S&P 500 or local equivalents) are intentionally excluded from the operational dataset at this stage. This filtering ensures that the Hierarchical Risk Parity (HRP) protocol and the latent feature extraction of the Denoising Autoencoder focus exclusively on the idiosyncratic and systemic risk structures of the individual constituent assets.

In [11]:
# Calculate continuous log returns
returns = np.log(raw_prices / raw_prices.shift(1)).dropna()
returns.tail(3)

,BAP PE Equity,SCCO PE EQUITY,CHILE CI EQUITY,SQM/B CI EQUITY,PFCIBEST CB EQUITY,CENCOSUD CI EQUITY,BSAN CI EQUITY,LTM CI EQUITY,BCI CI EQUITY,CIBEST CB EQUITY,...,IAM CI EQUITY,ILC CI EQUITY,CAP CI EQUITY,CORFICOL CB EQUITY,SALFACOR CI EQUITY,FORUS CI EQUITY,SONDA CI EQUITY,OROB CI EQUITY,VOLCABC1 PE EQUITY,BICE CI EQUITY
Dates,,,,,,,,,,,,,,,,,,,,,
2026-04-09,0.006537,0.023184,0.011638,-0.017940,-0.009968,0.007843,0.006431,0.009291,0.024693,0.005131,...,-0.007770,0.000000,-0.022489,0.011991,-0.000075,0.000000,-0.003377,0.012647,0.014389,0.001848
2026-04-10,-0.011396,0.002702,0.005855,0.005289,-0.006207,0.025835,0.006645,0.001260,0.017808,0.011563,...,-0.007135,0.007220,-0.000734,-0.004779,-0.013272,-0.004272,0.017661,0.021315,0.000000,0.036795
2026-04-13,0.024623,0.031200,-0.005506,0.057309,0.026910,-0.023884,0.000764,-0.009703,-0.005012,0.013929,...,0.002123,-0.026733,0.019630,0.000000,-0.014605,-0.016607,-0.006206,0.026876,-0.014389,0.031679


## 2. Algorithmic Core: Enhanced Hierarchical Risk Parity (HRP)

The classical HRP framework designed by Marcos López de Prado bypasses traditional quadratic optimization constraints by leveraging hierarchical clustering logic. The functions below formalize the exact matrix tree-clustering, inverse-variance optimization, and recursive bisection steps required to allocate portfolio weights dynamically.

### Hierarchical order of assets based on linkage (Quasi-diagonalization)

In [14]:
def get_quasi_diag(link):
    """
    Modern and robust pure-Python implementation of tree serialization 
    for Hierarchical Risk Parity. Bypasses pandas version compatibility bugs.
    """
    link = link.astype(int)
    sort_ix = [link[-1, 0], link[-1, 1]]
    num_items = link[-1, 3]  # Total number of leaves/assets
    
    # Recursively traverse the linkage tree from top to bottom
    while max(sort_ix) >= num_items:
        next_ix = []
        for i in sort_ix:
            if i >= num_items:
                # If it's an internal cluster node, extract its two children
                next_ix.extend([link[i - num_items, 0], link[i - num_items, 1]])
            else:
                # If it's a leaf node (asset), keep it
                next_ix.append(i)
        sort_ix = next_ix
        
    return sort_ix

### Inverse variance for a subcluster with stability correction

In [16]:
def getIVP(cov, **kargs):
    ivp = 1.0 / np.diag(cov)
    ivp /= ivp.sum()
    return ivp

### Variance of an asset subcluster

In [18]:
def getClusterVar(cov, cItems):
    cov_ = cov.loc[cItems, cItems]
    w_ = getIVP(cov_)
    cVar = np.dot(np.dot(w_, cov_), w_)
    return cVar

### Recursive Bisection to calculate HRP weights with stability correction

In [20]:
def getRecBipart(cov, sortIX):
    w = pd.Series(1.0, index=sortIX)
    cItems = [sortIX]
    while len(cItems) > 0:
        cItems = [i[j:k] for i in cItems for j, k in ((0, len(i) // 2), (len(i) // 2, len(i))) if len(i) > 1]
        for i in range(0, len(cItems), 2):
            cItems0 = cItems[i]
            cItems1 = cItems[i+1]
            cVar0 = getClusterVar(cov, cItems0)
            cVar1 = getClusterVar(cov, cItems1)
            alpha = 1.0 - cVar0 / (cVar0 + cVar1)
            w[cItems0] *= alpha
            w[cItems1] *= (1.0 - alpha)
    return w

### Dynamic calculation of correlation, distances, and linkage to ensure independence

In [22]:
def get_hrp_order_from_cov(cov_matrix):
    corr = cov_matrix.corr()
    dist = np.sqrt(0.5 * (1.0 - corr))
    dist = np.clip(dist, 0.0, 1.0)
    for i in range(dist.shape[0]):
        dist.iloc[i, i] = 0.0
    from scipy.spatial.distance import squareform
    link = linkage(squareform(dist.values), method='single')
    return get_quasi_diag(link), link

### Complete HRP weight pipeline

In [24]:
def calculate_hrp_weights(cov_matrix):
    quasi_diag_order, _ = get_hrp_order_from_cov(cov_matrix)
    asset_order = [cov_matrix.columns[i] for i in quasi_diag_order]
    hrp_weights = getRecBipart(cov_matrix, asset_order)
    return hrp_weights.reindex(cov_matrix.columns)

## 3. Core Backtesting Architecture: Dynamic Rolling Window Engine

To fully test our core hypotheses out-of-sample (**H1: Portfolio Stability/Turnover**, **H2: Risk-Adjusted Performance**, and **H3: Downside Risk Mitigation**), we deploy a simulation framework that mimics live institutional execution.

### Backtest Mechanics
We implement a **Rolling Window Strategy**:
* **Calibration Window ($T_{train} = 252$ days):** At each rebalancing date, the prior year of market log returns is ingested to dynamically retrain the standardization scaler, completely re-optimize the Autoencoder neural network from scratch, and recalculate the covariance structures.
* **Rebalancing Frequency ($\Delta t = 21$ days):** The portfolio's asset allocations are completely updated every calendar month.
* **Out-of-Sample Tracking:** Portfolio returns are recorded strictly within the subsequent test window ($t + \Delta t$), ensuring that performance metrics are completely isolated from look-ahead bias or in-sample overfitting.

## 4. Dynamic Rolling Window Backtesting Loop Execution

The following cell initializes the tracking structures and executes the formal walk-forward optimization pipeline, generating dynamic asset allocations under the empirical baseline and the adaptive Autoencoder filter.

In [27]:
# Backtest Engine Operational Hyperparameters
window_size = 252          # 1 trading year
rebalance_freq = 21        # Monthly rebalance
risk_free_annual = 0.0462

# Historical tracking containers
history_weights_emp = {}
history_weights_ae = {}

out_of_sample_returns = {
    'Empirical': [],
    'AE': []
}

print("[WALK-FORWARD BACKTEST ENGINE INITIALIZATION RUNNING]")

# WALK-FORWARD EXECUTION LOOP
for t in range(window_size, len(returns), rebalance_freq):

    # Define rolling train/test windows
    train_window = returns.iloc[t - window_size : t]
    test_window = returns.iloc[t : t + rebalance_freq]

    if len(test_window) == 0:
        break

    # ========================================================
    # PHASE A: Classical HRP Empirical Baseline
    # ========================================================
    cov_emp_t = train_window.cov()
    w_emp_t = calculate_hrp_weights(cov_emp_t)
    history_weights_emp[test_window.index[0]] = w_emp_t

    # ========================================================
    # PHASE B: WALK-FORWARD AUTOENCODER PIPELINE
    # ========================================================
    # CRITICAL FIX: Clear Keras global graph memory to prevent massive slowdowns
    tf.keras.backend.clear_session()
    
    # REPRODUCIBILITY LOCK: Force seed isolation inside the loop iteration
    tf.random.set_seed(42)

    scaler_t = StandardScaler()
    scaled_train = scaler_t.fit_transform(train_window)

    input_dim_t = scaled_train.shape[1]
    encoding_dim_t = max(2, input_dim_t // 2)

    input_layer_t = Input(shape=(input_dim_t,), name='WF_Input_Layer')

    encoded_t = Dense(
        encoding_dim_t,
        activation='relu',
        kernel_initializer=tf.keras.initializers.GlorotUniform(seed=42),
        name='WF_Encoder'
    )(input_layer_t)

    decoded_t = Dense(
        input_dim_t,
        activation='linear',
        kernel_initializer=tf.keras.initializers.GlorotUniform(seed=42),
        name='WF_Decoder'
    )(encoded_t)

    autoencoder_t = Model(inputs=input_layer_t, outputs=decoded_t, name='WF_Autoencoder')
    autoencoder_t.compile(optimizer='adam', loss='mse')

    early_stopping_t = EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=0
    )

    history_t = autoencoder_t.fit(
        scaled_train, scaled_train,
        epochs=150,
        batch_size=32,
        shuffle=True,
        validation_split=0.1,
        callbacks=[early_stopping_t],
        verbose=0
    )

    reconstructed_scaled_t = autoencoder_t.predict(scaled_train, verbose=0)

    reconstructed_train_t = pd.DataFrame(
        scaler_t.inverse_transform(reconstructed_scaled_t),
        index=train_window.index,
        columns=train_window.columns
    )

    cov_ae_t = reconstructed_train_t.cov()
    w_ae_t = calculate_hrp_weights(cov_ae_t)
    history_weights_ae[test_window.index[0]] = w_ae_t

    # ========================================================
    # PHASE C: OUT-OF-SAMPLE PERFORMANCE
    # ========================================================
    ret_emp_oos = (test_window * w_emp_t).sum(axis=1)
    ret_ae_oos = (test_window * w_ae_t).sum(axis=1)

    out_of_sample_returns['Empirical'].extend(ret_emp_oos)
    out_of_sample_returns['AE'].extend(ret_ae_oos)

[WALK-FORWARD BACKTEST ENGINE INITIALIZATION RUNNING]


## 5. Out-of-Sample Data Structuring & Alignment

To audit the models, the generated weight dictionaries and backtest return traces are converted into unified, time-aligned Pandas DataFrames, matching their corresponding historical market timelines.

In [29]:
# Transpose historical weight matrices to time-series DataFrames
df_weights_emp = pd.DataFrame(history_weights_emp).T
df_weights_ae = pd.DataFrame(history_weights_ae).T

# Extract and align exact out-of-sample trading dates
oos_dates = returns.index[
    window_size : 
    window_size + len(out_of_sample_returns['Empirical'])
]

# Consolidate final out-of-sample tracking returns
df_oos_returns = pd.DataFrame(
    out_of_sample_returns, 
    index=oos_dates
)

print("[WALK-FORWARD BACKTEST SIMULATION SUCCESSFULLY CONCLUDED]")

[WALK-FORWARD BACKTEST SIMULATION SUCCESSFULLY CONCLUDED]


## 6. Empirical Statistical Verification: Performance & Stability Audit

To draw rigorous mathematical conclusions regarding the dynamic integration of Deep Learning with the Hierarchical Risk Parity framework, we process the out-of-sample walk-forward backtest traces through five core operational metrics aligned with our research hypotheses:

* **Annualized Volatility:** Audits the structural variance expansion or compression induced by the non-linear filtering process.
* **Maximum Peak-to-Trough Drawdown:** Tracks the portfolio's tail-risk resilience and structural capital protection during periods of intense downside market stress.
* **Annualized Sharpe Ratio:** Evaluates risk-adjusted excess performance relative to the annualized risk-free rate milestone.
* **Mean Portfolio Turnover:** Calculates the average relative shifting of asset weights per rebalancing node, serving as a direct indicator of transaction-cost viability.
* **Mean Effective Number of Assets ($N_{eff}$):** Measures the true non-linear diversification breath, auditing whether the network prunes redundant allocation paths to focus on latent risk factors.

In [31]:
# 1. Performance Metric Mathematical Functions Definition
def calc_sharpe_ratio(returns_series, risk_free_rate=0.0, periods=252):
    excess_returns = returns_series - (risk_free_rate / periods)
    return np.sqrt(periods) * (excess_returns.mean() / excess_returns.std())

def calc_max_drawdown(returns_series):
    # Rigorous transformation assuming standard log-return time series aggregation
    cum_returns = np.exp(returns_series.cumsum())
    rolling_max = cum_returns.cummax()
    drawdown = (cum_returns - rolling_max) / rolling_max
    return drawdown.min()

def calc_turnover(weights_history_df):
    weight_changes = weights_history_df.diff().abs().dropna()
    turnover_per_step = weight_changes.sum(axis=1) / 2.0
    return turnover_per_step.mean()

def calc_neff(weights_vector):
    hhi = np.sum(weights_vector ** 2)
    return 1.0 / hhi if hhi > 0 else np.nan

# 2. Compile Cross-Model Historical Audit Log Table
results_summary = {
    'Operational Metric': [
        'Annualized Volatility', 
        'Maximum Peak-to-Trough Drawdown', 
        'Annualized Sharpe Ratio', 
        'Mean Portfolio Turnover',
        'Mean Effective Number of Assets'
    ],
    'HRP Empirical Baseline': [
        df_oos_returns['Empirical'].std() * np.sqrt(252),
        calc_max_drawdown(df_oos_returns['Empirical']),
        calc_sharpe_ratio(df_oos_returns['Empirical'], risk_free_annual),
        calc_turnover(df_weights_emp),
        df_weights_emp.apply(calc_neff, axis=1).mean()
    ],
    'HRP AE Framework': [  # Consolidated structural label (AE)
        df_oos_returns['AE'].std() * np.sqrt(252),  
        calc_max_drawdown(df_oos_returns['AE']),     
        calc_sharpe_ratio(df_oos_returns['AE'], risk_free_annual), 
        calc_turnover(df_weights_ae),               
        df_weights_ae.apply(calc_neff, axis=1).mean() 
    ]
}

# Construct and print final analytical metrics summary
df_hypothesis_results = pd.DataFrame(results_summary).set_index('Operational Metric')
print("\n" + "="*70)
print("     OUT-OF-SAMPLE EMPIRICAL HYPOTHESIS VALIDATION MATRIX")
print("="*70)
import datetime
from IPython.display import display
display(df_hypothesis_results.round(4))


     OUT-OF-SAMPLE EMPIRICAL HYPOTHESIS VALIDATION MATRIX


,HRP Empirical Baseline,HRP AE Framework
Operational Metric,,
Annualized Volatility,0.1165,0.1174
Maximum Peak-to-Trough Drawdown,-0.1192,-0.1142
Annualized Sharpe Ratio,1.3702,1.5673
Mean Portfolio Turnover,0.1390,0.1881
Mean Effective Number of Assets,26.6196,23.3011


## 13. General Conclusion and Discussion

In conclusion, the out-of-sample empirical validation under a dynamic rolling-window framework demonstrates a highly valuable structural trade-off when integrating an Autoencoder (AE) into the Hierarchical Risk Parity (HRP) protocol. Unlike static filtering architectures, the adaptive walk-forward optimization regime implemented in this framework allows the HRP-AE protocol to continuously re-estimate non-linear latent co-dependencies, effectively adapting to shifting market regimes and isolating estimation noise from the covariance structure.

The empirical results conclusively validate the framework's capacity to unlock superior risk-adjusted performance out-of-sample. The HRP-AE model achieved a massive expansion in the annualized Sharpe Ratio, rising from 1.3702 in the empirical baseline to an exceptional 1.5673. This substantial alpha generation behavior operates through an active tactical restructuring of the portfolio's topology. Under this dynamic regime, the network prioritizes return-to-risk optimization over passive turnover minimization; consequently, the continuous re-optimization pipeline leads to a controlled increase in mean portfolio turnover (moving from 0.1390 to 0.1881) and a strategic concentration in the mean effective number of assets ($N_{eff}$ shifting from 26.6196 to 23.3011). Furthermore, while this active reallocation target induces a marginal expansion in annualized volatility (from 0.1165 to 0.1174), it achieves a structural mitigation of tail risk, successfully compressing the maximum peak-to-trough drawdown from -11.92% down to -11.42%.

Consequently, the HRP-AE framework establishes itself as an exceptionally powerful and responsive asset allocation engine for institutional managers. By successfully exchanging minor transactional and turnover efficiencies for a massive leap in risk-adjusted performance and enhanced drawdown protection, the architecture proves that deep learning latent representations can systematically exploit non-linear correlation structures and structural breakthroughs where traditional linear benchmarks fall short.

## References

<p style="padding-left: 30px; text-indent: -30px;">Goodfellow, I., Bengio, Y., & Courville, A. (2016). Deep learning. MIT Press. https://www.deeplearningbook.org/</p>

<p style="padding-left: 30px; text-indent: -30px;">López de Prado, M. (2016). Building diversified portfolios that outperform out of sample. The Journal of Portfolio Management, 42(4), 59–69. https://doi.org/10.3905/jpm.2016.42.4.059</p>

<p style="padding-left: 30px; text-indent: -30px;">Ramírez-Carrillo, G. (2025). Hierarchical risk parity with nonlinear latent covariance filtering. arXiv. https://arxiv.org/abs/2509.03712</p>

## Academic Disclaimer

This material was developed exclusively for academic research and educational purposes.

The results presented in this notebook should not be interpreted as investment advice, portfolio recommendations, or production-grade financial deployment strategies.

All experiments are subject to:
- sample dependency,
- model specification risk,
- and market regime sensitivity.

**Further robustness validation is required before any real-world implementation.**